[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Full-Text Search &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/logbook.db` with the logbook, as the notebook's Setup did, and creates
what the worked examples created: `search`, `stemmed`, and `logbook_index` with its three triggers.
It defines `count` and `as_words`. Run it first. The tasks do not depend on one another, and the last
cell closes the connection and removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "logbook.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
EVENTS = [
    "Heater on the sensor mast checked and working.",
    "Battery replaced after a low voltage warning.",
    "Snow cleared from the rain gauge.",
    "Sensor recalibrated against the reference thermometer.",
    "Ice on the anemometer, so the wind readings for the morning are unreliable.",
    "Annual service of the station completed.",
    "Fence repaired after a storm.",
    "Data logger restarted after a power cut, and no readings were lost.",
    "Heaters on the mast replaced.",
    "Visited twice to check the heater.",
]
ON_WARM_DAYS = {EVENTS[2]: "Grass cut around the rain gauge.", EVENTS[4]: "Anemometer bearings greased."}


def year_of_readings():
    """Every hour of 2025 at the four stations, as Why sqlite3 made them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


def logbook():
    """A technician's note for every station and day of 2025, written from that day's readings."""
    days = {}
    for station, hour, celsius in year_of_readings():
        days.setdefault((hour[:10], station), []).append(celsius)
    for (day, station), temperatures in sorted(days.items()):
        known = [celsius for celsius in temperatures if celsius is not None]
        if not known:
            yield station, day, "Data logger failed overnight, and there are no readings for the whole day."
            continue
        low, high = min(known), max(known)
        if high < 0:
            weather = f"Frost all day, between {low} and {high} degrees."
        elif low < 0:
            weather = f"Night frost down to {low} degrees, and a thaw to {high} by the afternoon."
        else:
            weather = f"Above freezing all day, between {low} and {high} degrees."
        day_of_year = datetime.strptime(day, "%Y-%m-%d").timetuple().tm_yday
        number = (day_of_year * 37 + list(STATIONS).index(station) * 101) % 23
        event = EVENTS[number] if number < len(EVENTS) else ""
        if low >= 0:
            event = ON_WARM_DAYS.get(event, event)
        yield station, day, f"{weather} {event}".strip()


build = sqlite3.connect(DATABASE)
build.execute("CREATE TABLE logbook (id INTEGER PRIMARY KEY, station TEXT NOT NULL, day TEXT NOT NULL, note TEXT NOT NULL)")
build.executemany("INSERT INTO logbook (station, day, note) VALUES (?, ?, ?)", logbook())
build.commit()
build.close()


conn = sqlite3.connect(DATABASE)
conn.executescript("""
    CREATE VIRTUAL TABLE search USING fts5(station, day UNINDEXED, note);
    INSERT INTO search (rowid, station, day, note) SELECT id, station, day, note FROM logbook;

    CREATE VIRTUAL TABLE stemmed USING fts5(note, tokenize = 'porter unicode61');
    INSERT INTO stemmed (rowid, note) SELECT id, note FROM logbook;

    CREATE VIRTUAL TABLE logbook_index USING fts5(note, content = 'logbook', content_rowid = 'id');
    INSERT INTO logbook_index (logbook_index) VALUES ('rebuild');
    CREATE TRIGGER logbook_after_insert AFTER INSERT ON logbook BEGIN
        INSERT INTO logbook_index (rowid, note) VALUES (new.id, new.note);
    END;
    CREATE TRIGGER logbook_after_delete AFTER DELETE ON logbook BEGIN
        INSERT INTO logbook_index (logbook_index, rowid, note) VALUES ('delete', old.id, old.note);
    END;
    CREATE TRIGGER logbook_after_update AFTER UPDATE ON logbook BEGIN
        INSERT INTO logbook_index (logbook_index, rowid, note) VALUES ('delete', old.id, old.note);
        INSERT INTO logbook_index (rowid, note) VALUES (new.id, new.note);
    END;
""")


def count(table, query):
    """How many rows of a full-text table match a query."""
    return conn.execute(f"SELECT COUNT(*) FROM {table} WHERE {table} MATCH ?", (query,)).fetchone()[0]


def as_words(text):
    """Text typed into a search box, as an FTS5 query that needs every word and treats none as an operator."""
    return " ".join('"' + word.replace('"', '""') + '"' for word in text.split())


print("notes:", conn.execute("SELECT COUNT(*) FROM logbook").fetchone()[0], "| saying degrees:", count("logbook_index", "degrees"))


notes: 1460 | saying degrees: 1459


**1.** The best matches for a battery or a power cut.


In [2]:
for day, station, score in conn.execute("""
    SELECT day, station, round(rank, 3) FROM search
    WHERE search MATCH 'battery OR power'
    ORDER BY rank, rowid
    LIMIT 3
"""):
    print(day, station, score)


2025-01-07 Svalbard -2.96
2025-01-08 Tromso -2.96
2025-01-30 Svalbard -2.96


`battery` and `power` never share a note here, so every match has one of the two words, and the
shortest notes, whose weather sentences are shortest, scored best. `rowid` orders the notes whose
scores are equal.


**2.** Snow near the gauge.


In [3]:
for distance in (2, 5):
    print(f"NEAR(snow gauge, {distance}):", count("search", f"NEAR(snow gauge, {distance})"))


NEAR(snow gauge, 2): 0
NEAR(snow gauge, 5): 33


In `Snow cleared from the rain gauge.`, four words stand between `snow` and `gauge`, so a distance of
two found nothing, and five found every such note. The distance goes into the query text with an
f-string because it is an integer this code chose, and the query language has no placeholders of its
own.


**3.** Frost at every station.


In [4]:
print(conn.execute("""
    SELECT station, COUNT(*) FROM search
    WHERE search MATCH 'note: frost'
    GROUP BY station
    ORDER BY COUNT(*) DESC, station
""").fetchall())


[('Svalbard', 306), ('Tromso', 182), ('Oslo', 143), ('Bergen', 120)]


`note:` keeps the word to the notes' text, so a station named Frost, if there were one, would not
count. `GROUP BY` works on the rows `MATCH` found as it does on any other rows, and Svalbard, the
coldest station, had frost on the most days.


**4.** `repair`, with and without stems.


In [5]:
print("search MATCH 'repair':", count("search", "repair"))
print("stemmed MATCH 'repair':", count("stemmed", "repair"))


search MATCH 'repair': 0
stemmed MATCH 'repair': 64


The notes say `Fence repaired after a storm.`, and never `repair`, so the exact index found nothing.
The `porter` tokenizer stored `repaired` as its stem, `repair`, and reduces the query to the same
stem, so the stemmed index found every one of those notes.


**5.** A note that comes and goes.


In [6]:
with conn:
    note_id = conn.execute("INSERT INTO logbook (station, day, note) VALUES (?, ?, ?) RETURNING id",
                           ("Tromso", "2025-12-31", "Aurora seen over the station at midnight.")).fetchone()[0]
print("after the insert:", count("logbook_index", "aurora"))

with conn:
    conn.execute("DELETE FROM logbook WHERE id = ?", (note_id,))
print("after the delete:", count("logbook_index", "aurora"))


after the insert: 1
after the delete: 0


The insert trigger added the note's words to the index, and the delete trigger took them out, giving
FTS5 the old text it needs for that. Neither statement touched `logbook_index` directly.


**6.** The most common words.


In [7]:
conn.execute("CREATE VIRTUAL TABLE search_words USING fts5vocab(search, 'row')")
for term, notes, occurrences in conn.execute("""
    SELECT term, doc, cnt FROM search_words ORDER BY doc DESC, term LIMIT 5
"""):
    print(f"{term:<10} in {notes} notes, {occurrences} times")


and        in 1460 notes, 1587 times
degrees    in 1459 notes, 1459 times
day        in 982 notes, 982 times
all        in 981 notes, 981 times
between    in 981 notes, 981 times


An `fts5vocab` table of type `'row'` has a row for every word in the index: `doc` is how many rows
contain it and `cnt` how many times it appears. The most common words are the ones every weather
sentence uses, which is why `bm25` gives them little weight.

Last, close the connection and remove the scratch folder:


In [8]:
conn.close()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Full-Text Search](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/16-full-text-search.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
